In [1]:
!pip install anthropic pillow


In [2]:
import anthropic, base64, os, json, time
print(f"anthropic: {anthropic.__version__}")
print("Imports OK")


anthropic: 0.96.0
Imports OK


# NeurIPS Computational Resources & Reproducibility Metadata

Required by **NeurIPS 2025 Paper Checklist §8**. Run to print a live hardware/version summary. Fill `[TODO]` fields before submission.

In [3]:
"""
NeurIPS 2025 Checklist S8 - Computational Resources
====================================================
Hardware
  GPU      : NVIDIA RTX A6000 (48 GB VRAM)
  CPU      : [TODO: e.g. AMD EPYC 7542 32-core]
  RAM      : [TODO: e.g. 256 GB DDR4]
  OS       : Windows 11 / Ubuntu 22.04
  Provider : Local on-premise workstation

Model & Inference
  Model      : claude-opus-4-7
  max_tokens : 4096 per call
  Temp       : 0.0 (Zero-Shot / Sequential / LtM / ReAct / CoT)
               0.1 (Iterative)
               0.1-0.5 (Self-Consistency runs)
               0.1/0.7 (Meta-Prompting: analysis/generation)

API Calls per Video  (C = ceil(frames / 10))
  Zero-Shot        : C
  Sequential       : 5*C + 1
  Least-To-Most    : 8*C + 1
  ReAct            : C + 1
  True Iterative   : up to 8*C
  Self-Consistency : 5*C + 1
  Meta-Prompting   : 2*C + 2
  Chain-of-Thought : C + 1
  Total/video (50 frames, C=5) ~ 162 calls ~ 243 000 tokens

Total Compute (fill before submission)
  Dataset  : [TODO] videos x [TODO] avg frames
  Calls    : [TODO] x 162 = [TODO]
  Tokens   : [TODO] x 243 000 = [TODO]
  Time     : ~[TODO] hours

Reproducibility
  - Checkpoint files save progress after every video (atomic write)
  - Re-running any cell after failure resumes with zero extra API cost
  - All raw API responses saved as JSON before post-processing
  - Chunk-level saves after every 10-frame batch
"""
import subprocess, platform, datetime
print("=" * 64)
print(f"  NeurIPS Compute Summary  - {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 64)
print(f"  Python  : {platform.python_version()}")
print(f"  OS      : {platform.platform()}")
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True).strip()
    for line in smi.split("\n"):
        print(f"  GPU     : {line.strip()}")
except Exception:
    print("  GPU     : nvidia-smi not available")
try:
    import anthropic
    print(f"  anthropic  : {anthropic.__version__}")
except Exception:
    pass
print(f"  Model   : claude-opus-4-7")
print(f"  Chunks  : 10 frames/chunk  |  max_tokens : 4096")
print(f"  Retry   : 7 attempts, exponential back-off, cap 5 min")
print("=" * 64)


  NeurIPS Compute Summary  - 2026-04-22 23:38
  Python  : 3.10.11
  OS      : Windows-10-10.0.26100-SP0
  GPU     : NVIDIA H100 NVL, 95830 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB
  anthropic  : 0.96.0
  Model   : claude-sonnet-4-6
  Chunks  : 10 frames/chunk  |  max_tokens : 4096
  Retry   : 7 attempts, exponential back-off, cap 5 min


#Iterative prompting
Iterative Prompting Approach
The Iterative Prompting technique follows a structured refinement process:

- Initial Analysis: The model provides a first-pass analysis of what appears to be happening in the frames
- Guided Iterations: Through a series of targeted follow-up prompts, the model refines specific aspects of its analysis
- Progressive Improvement: Each round builds on previous insights while addressing potential weaknesses or gaps
- Final Synthesis: After multiple refinement rounds, the model creates a final, comprehensive assessment

Implementation Highlights

Multi-Round Refinement:

Starts with an initial general analysis prompt
Follows with 4 specialized refinement rounds:

- People and relationships focus
- Actions and intent focus
- Criminal elements and evidence focus
- Critical examination (missing elements, alternative interpretations)


Concludes with a final synthesis prompt for each chunk


Complete Frame Processing:

- Processes all frames in chunks of 10 frames each
- Each chunk undergoes the full iterative process independently


Conversation Continuity:

- Maintains the complete conversation history throughout all rounds
- Each refinement builds on the accumulated context from previous rounds
- Creates a progressive improvement cycle where later responses incorporate earlier insights


Holistic Synthesis:

- After all chunks are iteratively analyzed, performs a final cross-chunk synthesis
- Creates a coherent narrative of the entire incident
- Addresses any discrepancies between chunk analyses

In [ ]:
import os
import json
import base64
import time
from datetime import datetime
from collections import defaultdict
import anthropic


import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
# ================================================================
#  CONFIGURATION  -  edit these paths to match your machine
# ================================================================
FRAMES_DIR     = r"C:\Opeyemi\PROMPTS\FRAMES"   # pre-extracted frames
RESULTS_BASE   = r"C:\Opeyemi\PROMPTS\RESULTS"  # all JSON outputs
FRAME_EXT      = ".jpg"
FRAME_INTERVAL = 1   # 1=every frame; 2=every other; etc.
MAX_WORKERS    = 8    # parallel videos processed at once (tune to API rate-limit)
BATCH_SIZE     = 20   # frames per API call (per-cell so cells run independently)

# ================================================================
#  FRAME HELPERS  -  self-contained in every cell
# ================================================================

def extract_frame_number(filename):
    """Return integer index from frame_00042.jpg style names."""
    import re as _re
    name = os.path.splitext(filename)[0]
    m = _re.search(r"frame[_\-]?(\d+)", name, _re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = _re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 0


def discover_all_videos_and_frames(frames_dir=None):
    """
    Walk FRAMES_DIR and return a manifest of all extracted videos.

    Expected layout (created by the Video-to-Frames extractor):
        FRAMES_DIR/
            Abuse/
                Abuse001_x264/
                    frame_00001.jpg
                    frame_00002.jpg ...
                Abuse002_x264/ ...
            Arrest/ ...   (13 UCF-Crime categories)

    Returns dict "<CrimeType>_<VideoStem>" -> {
        "crime_type": "Abuse",
        "video_id":   "Abuse001_x264",
        "frames_dir": r"C:\...\FRAMES\Abuse\Abuse001_x264",
        "frames":     ["frame_00001.jpg", ...]   # sorted by number
    }
    """
    if frames_dir is None:
        frames_dir = FRAMES_DIR
    print(f"\n=== DISCOVERING FRAMES ===")
    print(f"    Root : {frames_dir}")
    all_videos = {}
    if not os.path.isdir(frames_dir):
        print(f"  ERROR: FRAMES_DIR not found: {frames_dir}")
        print("  Run the Video-to-Frames extractor first, or check the path.")
        return all_videos
    crime_types = sorted([
        d for d in os.listdir(frames_dir)
        if os.path.isdir(os.path.join(frames_dir, d)) and not d.startswith("_")
    ])
    print(f"  Categories : {crime_types}")
    for crime_type in crime_types:
        cat_dir = os.path.join(frames_dir, crime_type)
        video_stems = sorted([
            d for d in os.listdir(cat_dir)
            if os.path.isdir(os.path.join(cat_dir, d))
        ])
        print(f"    {crime_type:20s}: {len(video_stems)} videos")
        for video_stem in video_stems:
            vdir = os.path.join(cat_dir, video_stem)
            frame_files = sorted(
                [ff for ff in os.listdir(vdir) if ff.lower().endswith(FRAME_EXT)],
                key=extract_frame_number
            )
            if not frame_files:
                print(f"      WARNING: no {FRAME_EXT} frames in {vdir} - skipping")
                continue
            key = f"{crime_type}_{video_stem}"
            all_videos[key] = {
                "crime_type" : crime_type,
                "video_id"   : video_stem,
                "frames_dir" : vdir,
                "frames"     : frame_files,
            }
    print(f"  Total videos ready: {len(all_videos)}")
    return all_videos


def load_frames_for_video(video_info, frame_interval=1):
    """Read every frame_interval-th .jpg, base64-encode, return ordered list of (filename, b64)."""
    vdir        = video_info["frames_dir"]
    frame_files = video_info["frames"]
    video_id    = video_info["video_id"]
    selected    = frame_files[::frame_interval]
    label = "ALL" if frame_interval == 1 else f"every {frame_interval}th"
    print(f"  Loading {len(selected)} frames ({label}) for {video_id} ...")
    frames_list = []
    for ff in selected:
        fp = os.path.join(vdir, ff)
        try:
            with open(fp, "rb") as fh:
                b64 = base64.b64encode(fh.read()).decode("utf-8")
            frames_list.append((ff, b64))
        except Exception as e:
            print(f"    ERROR loading {ff}: {e}")
    print(f"  Loaded {len(frames_list)}/{len(selected)} frames OK")
    return frames_list   # list of (filename, b64_string)

SAVE_DIR = r"C:\Opeyemi\PROMPTS\RESULTS\CLAUDE\TRUE-ITERATIVE"
os.makedirs(SAVE_DIR, exist_ok=True)


CHECKPOINT_FILE = os.path.join(SAVE_DIR, "true_iterative_checkpoint.json")


def load_checkpoint():
    """Resume from last completed video after any crash or network failure."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r") as f:
                data = json.load(f)
            n = len(data.get("completed_videos", []))
            print(f"  Checkpoint: {n} videos already done - skipping them.")
            return data
        except Exception as e:
            print(f"  Could not read checkpoint ({e}) - starting fresh.")
    return {"completed_videos": [], "results": {}}


def save_checkpoint(data):
    """Atomic write so the file is never corrupted on a crash."""
    os.makedirs(SAVE_DIR, exist_ok=True)
    tmp = CHECKPOINT_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)







def make_claude_request_robust(client, model_name, messages,
                               system=None, temperature=0.1,
                               max_retries=7, base_wait=5):
    """
    Fault-tolerant Claude API call.
    Retries on: RateLimitError, APIConnectionError (network drop/DNS),
                APITimeoutError, and 5xx server errors.
    Uses exponential back-off with jitter.
    Returns an error string on permanent failure so the caller can save it
    and move on rather than crashing the whole run.
    """
    import random, anthropic

    RETRYABLE = (
        anthropic.RateLimitError,
        anthropic.APIConnectionError,
        anthropic.APITimeoutError,
    )
    PERMANENT = (
        anthropic.AuthenticationError,
        anthropic.PermissionDeniedError,
        anthropic.NotFoundError,
    )

    for attempt in range(1, max_retries + 1):
        try:
            kwargs = dict(
                model=model_name,
                max_tokens=4096,
                messages=messages,
            )
            if system:
                kwargs["system"] = system
            response = client.messages.create(**kwargs)
            return response.content[0].text

        except PERMANENT as e:
            msg = f"FATAL_ERROR: {type(e).__name__}: {e}"
            print(f"[FATAL] {msg}  -- will not retry.")
            return msg

        except RETRYABLE as e:
            wait = min(base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2), 300)
            print(f"[Retry {attempt}/{max_retries}] {type(e).__name__}: {e}")
            print(f"  Waiting {wait:.1f}s before next attempt ...")
            time.sleep(wait)

        except anthropic.APIStatusError as e:
            if e.status_code >= 500:
                wait = min(base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2), 300)
                print(f"[Retry {attempt}/{max_retries}] HTTP {e.status_code}: {e}")
                print(f"  Waiting {wait:.1f}s ...")
                time.sleep(wait)
            else:
                msg = f"FATAL_ERROR: HTTP_{e.status_code}: {e}"
                print(f"[FATAL] {msg}  -- will not retry.")
                return msg

        except Exception as e:
            if attempt < max_retries:
                print(f"[Retry {attempt}/{max_retries}] Unexpected {type(e).__name__}: {e}")
                time.sleep(base_wait * attempt)
            else:
                return f"ERROR: {type(e).__name__}: {e}"

    return f"ERROR: All {max_retries} attempts exhausted"





class TrueIterativeClaudeAnalyzer:
    """
    True iterative prompting: a single core question is asked, the answer is
    critiqued, and the question is refined with the critique fed back in.
    The loop continues until convergence (consecutive answers are very
    similar) or max_iterations is reached.

    Convergence is measured by Jaccard similarity of token sets between
    consecutive answers, which is good enough for "did the model stop
    changing its mind" without needing an embedding model.
    """

    MODEL = "claude-opus-4-7"

    SYSTEM_PROMPT = (
        "You are an expert forensic video analyst specializing in crime detection "
        "and security surveillance. You analyze video frames methodically, noting "
        "details about people, actions, environment, and potential criminal activity. "
        "When refining an earlier answer based on a critique, integrate the critique "
        "honestly: keep what was correct, fix what was wrong, and explicitly say what "
        "changed and why."
    )

    CORE_QUESTION = (
        "Analyze these video frames for criminal activity. What crime is occurring, "
        "who is involved, what evidence supports your conclusion, and how confident "
        "are you in this assessment? Classify the activity as one of: Abuse, Arrest, "
        "Arson, Assault, Burglary, Explosion, Fighting, RoadAccidents, Robbery, "
        "Shooting, Shoplifting, Stealing, Vandalism, or Normal."
    )

    def __init__(self, api_key: str, batch_size: int = BATCH_SIZE):
        self.client             = anthropic.Anthropic(api_key=api_key)
        self.batch_size         = batch_size
        self.max_iterations     = 5     # hard cap to prevent runaway cost
        self.convergence_thresh = 0.85  # Jaccard >= this means converged

    # ----------------------------------------------------------
    def _build_image_blocks(self, batch: list) -> list:
        blocks = []
        for fname, b64 in batch:
            blocks.append({
                "type": "image",
                "source": {
                    "type":       "base64",
                    "media_type": "image/jpeg",
                    "data":       b64,
                },
            })
            blocks.append({"type": "text", "text": f"[Frame: {fname}]"})
        return blocks

    # ----------------------------------------------------------
    @staticmethod
    def _jaccard(a: str, b: str) -> float:
        ta = set(a.lower().split())
        tb = set(b.lower().split())
        if not ta and not tb: return 1.0
        if not ta or not tb:  return 0.0
        return len(ta & tb) / len(ta | tb)

    # ----------------------------------------------------------
    def _initial_answer(self, batch, batch_num, total_batches):
        image_blocks = self._build_image_blocks(batch)
        framing = (
            f"Batch {batch_num}/{total_batches}. "
            f"This is iteration 1 (initial answer).\n\n"
            f"{self.CORE_QUESTION}"
        )
        messages = [{"role": "user",
                     "content": image_blocks + [{"type": "text", "text": framing}]}]
        return make_claude_request_robust(
            self.client, self.MODEL, messages, system=self.SYSTEM_PROMPT)

    def _critique(self, batch, batch_num, total_batches, prior_answer, iteration):
        image_blocks = self._build_image_blocks(batch)
        framing = (
            f"Batch {batch_num}/{total_batches}. "
            f"You are critiquing an analysis at iteration {iteration}.\n\n"
            f"PRIOR ANALYSIS:\n{prior_answer}\n\n"
            "Your task: identify any missing observations, unsupported claims, "
            "ambiguous reasoning, or wrong identifications in the prior analysis. "
            "Be specific and concrete. Do NOT rewrite the analysis \u2014 just produce a "
            "bulleted critique."
        )
        messages = [{"role": "user",
                     "content": image_blocks + [{"type": "text", "text": framing}]}]
        return make_claude_request_robust(
            self.client, self.MODEL, messages, system=self.SYSTEM_PROMPT)

    def _refine(self, batch, batch_num, total_batches, prior_answer, critique, iteration):
        image_blocks = self._build_image_blocks(batch)
        framing = (
            f"Batch {batch_num}/{total_batches}. Iteration {iteration} (refinement).\n\n"
            f"PRIOR ANALYSIS:\n{prior_answer}\n\n"
            f"CRITIQUE OF PRIOR ANALYSIS:\n{critique}\n\n"
            f"ORIGINAL QUESTION:\n{self.CORE_QUESTION}\n\n"
            "Produce a REFINED analysis that addresses the critique. Keep correct "
            "parts of the prior analysis; fix what the critique flagged. End with "
            "the same classification fields (crime type, confidence, evidence)."
        )
        messages = [{"role": "user",
                     "content": image_blocks + [{"type": "text", "text": framing}]}]
        return make_claude_request_robust(
            self.client, self.MODEL, messages, system=self.SYSTEM_PROMPT)

    # ----------------------------------------------------------
    def _iterate_one_batch(self, batch, batch_num, total_batches):
        """Run the iterative critique-refine loop on a single batch."""
        history = []  # list of dicts: {iteration, answer, critique, jaccard_to_prev}
        answer = self._initial_answer(batch, batch_num, total_batches)
        history.append({"iteration": 1, "answer": answer,
                        "critique": None, "jaccard_to_prev": None})

        for it in range(2, self.max_iterations + 1):
            critique = self._critique(batch, batch_num, total_batches, answer, it)
            refined  = self._refine(batch, batch_num, total_batches,
                                    answer, critique, it)
            jac = self._jaccard(answer, refined)
            history.append({"iteration": it, "answer": refined,
                            "critique": critique, "jaccard_to_prev": jac})
            answer = refined
            if jac >= self.convergence_thresh:
                break

        return history

    # ----------------------------------------------------------
    def analyze_frames(self, frames_list: list, video_id: str, crime_type: str) -> dict:
        total_frames  = len(frames_list)
        batches       = [frames_list[i:i + self.batch_size]
                         for i in range(0, total_frames, self.batch_size)]
        total_batches = len(batches)
        print(f"\n  [Iterative] {video_id} | {total_frames} frames | "
              f"{total_batches} batches of up to {self.batch_size}")

        per_batch_history = []
        for b_idx, batch in enumerate(batches, start=1):
            print(f"    Batch {b_idx}/{total_batches} ...")
            hist = self._iterate_one_batch(batch, b_idx, total_batches)
            print(f"      converged at iter {hist[-1]['iteration']}/"
                  f"{self.max_iterations} (jaccard={hist[-1]['jaccard_to_prev']})")
            per_batch_history.append({"batch_num": b_idx, "history": hist})

        # Final synthesis: combine each batch's final-iteration answer
        joined = "\n\n".join(
            f"--- Batch {h['batch_num']} final iteration ---\n{h['history'][-1]['answer']}"
            for h in per_batch_history
        )
        synth_prompt = (
            f"You finished iterative refinement on {total_batches} batches "
            f"covering {total_frames} frames of one security video. Below is each "
            f"batch's final converged analysis:\n\n{joined}\n\n"
            "Produce a FINAL combined report:\n"
            "1. SCENE DESCRIPTION (across all batches).\n"
            "2. CRIME CLASSIFICATION (one of Abuse, Arrest, Arson, Assault, Burglary, "
            "Explosion, Fighting, RoadAccidents, Robbery, Shooting, Shoplifting, "
            "Stealing, Vandalism, Normal).\n"
            "3. CONFIDENCE (0-100%).\n"
            "4. KEY EVIDENCE.\n"
            "5. RECOMMENDED ACTION."
        )
        final_report = make_claude_request_robust(
            self.client, self.MODEL,
            [{"role": "user", "content": synth_prompt}],
            system=self.SYSTEM_PROMPT)

        return {
            "video_id":             video_id,
            "crime_type":           crime_type,
            "frames_analyzed":      total_frames,
            "total_batches":        total_batches,
            "batch_size":           self.batch_size,
            "prompting_technique":  "TRUE-ITERATIVE",
            "model":                self.MODEL,
            "max_iterations":       self.max_iterations,
            "convergence_thresh":   self.convergence_thresh,
            "timestamp":            time.strftime("%Y-%m-%d %H:%M:%S"),
            "batch_iterations":     per_batch_history,
            "final_analysis":       final_report,
        }


def process_all_crime_folders(api_key):
    """
    Analyse every video in FRAMES_DIR.
    Already-completed videos are skipped (checkpoint/resume) so re-running
    after any failure picks up exactly where it stopped.
    """
    analyzer   = TrueIterativeClaudeAnalyzer(api_key)
    all_videos = discover_all_videos_and_frames()
    if not all_videos:
        print("No videos found! Verify FRAMES_DIR path."); return {}
    cp          = load_checkpoint()
    all_results = cp.get("results", {})
    done_set    = set(cp.get("completed_videos", []))
    skipped     = []
    total       = len(all_videos)
    remaining   = {k: v for k, v in all_videos.items() if k not in done_set}
    print(f"\nVideos: total={total} | done={len(done_set)} | remaining={len(remaining)}")
    # ================================================================
    #  PARALLEL VIDEO PROCESSING
    # ================================================================
    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(vkey, vinfo):
        """Process a single video in its own thread. Checkpoint writes are locked."""
        with print_lock:
            print(f"\n  [START] {vkey}")
        try:
            frames = load_frames_for_video(vinfo, FRAME_INTERVAL)
            if not frames:
                return vkey, None, "no frames"
            res = analyzer.analyze_frames(frames, vinfo["video_id"], vinfo["crime_type"])
            with checkpoint_lock:
                all_results[vkey] = res
                done_set.add(vkey)
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            with print_lock:
                print(f"  [DONE]  {vkey}  ({len(done_set)}/{total})")
            return vkey, res, None
        except Exception as e:
            with print_lock:
                print(f"  [ERROR] {vkey}: {e}")
            with checkpoint_lock:
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            return vkey, None, f"error: {e}"

    print(f"\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_process_one_video, k, v) for k, v in remaining.items()]
        for fut in as_completed(futures):
            vkey, _, err = fut.result()
            if err:
                skipped.append(f"{vkey} ({err})")

    ts = time.strftime("%Y%m%d_%H%M%S")
    summary = os.path.join(SAVE_DIR, f"true_iterative_summary_{ts}.json")
    with open(summary, "w") as f:
        json.dump(all_results, f, indent=2)
    if skipped:
        with open(os.path.join(SAVE_DIR, f"skipped_{ts}.txt"), "w") as f:
            f.write("\n".join(skipped))
    print(f"\nDone. Summary -> {summary}")
    print(f"  Processed: {len(all_results)} | Skipped: {len(skipped)}")
    return all_results


def test_claude_api(api_key):
    """Test Claude API connection"""
    print("Testing Claude API connection...")

    try:
        client = anthropic.Anthropic(api_key=api_key)

        response = client.messages.create(
            model="claude-opus-4-7",
            max_tokens=100,
            messages=[
                {
                    "role": "user",
                    "content": "Hello, can you respond with 'API connection successful'?"
                }
            ]
        )

        response_text = response.content[0].text
        print("✓ Claude API connection successful!")
        print(f"Response: {response_text}")
        return True

    except anthropic.APIError as e:
        print(f"✗ API Error: {str(e)}")
        return False
    except anthropic.AuthenticationError as e:
        print(f"✗ Authentication Error: {str(e)}")
        print("Please check your API key is valid and has sufficient credits.")
        return False
    except anthropic.RateLimitError as e:
        print(f"✗ Rate Limit Error: {str(e)}")
        return False
    except Exception as e:
        print(f"✗ Connection error: {str(e)}")
        return False

def check_authentication():
    """Placeholder function to check authentication"""
    return True

def run():
    # Load API key from file
    with open(r"C:\Opeyemi\PROMPTS\API-KEYS\claude.txt", "r") as _f:
        api_key = _f.read().strip()

    """Main execution function"""
    print("TRUE Iterative Prompting Crime Video Analysis with Claude Sonnet 4 - ALL Frames")
    print("="*85)
    print("TRUE ITERATIVE = Same question refined repeatedly until convergence")
    print("="*85)

    # Test directory access first
    print("Testing directory access...")
    for path in [FRAMES_DIR, SAVE_DIR]:
        print(f"Path: {path}")
        print(f"  Exists: {os.path.exists(path)}")
        if os.path.exists(path):
            try:
                contents = os.listdir(path)
                print(f"  Contains {len(contents)} items")
                if contents:
                    print(f"  First few items: {contents[:3]}")
            except Exception as e:
                print(f"  Error accessing contents: {str(e)}")

    # Get API key
    try:
        api_key_path = "C:\Opeyemi\PROMPTS\API-KEYS\claude.txt"
        print(f"Trying to load API key from: {api_key_path}")
        print(f"File exists: {os.path.exists(api_key_path)}")

        with open(api_key_path, "r") as f:
            api_key = f.read().strip()

        if not api_key:
            print("✗ Failed to load Claude API key: File is empty")
            return

        print("✓ Successfully loaded Claude API key")
        print(f"API key starts with: {api_key[:10]}...")

    except Exception as e:
        print(f"✗ Failed to load Claude API key: {str(e)}")
        return

    # Test Claude API connection
    if not test_claude_api(api_key):
        print("✗ Claude API test failed. Please check your API key and connection.")
        return

    # Check authentication
    if not check_authentication():
        print("✗ Authentication not completed.")
        return

    # Verify directories exist
    print("\nVerifying directories:")
    print(f"Data directory exists: {os.path.exists(FRAMES_DIR)}")
    print(f"Save directory exists: {os.path.exists(SAVE_DIR)}")

    if not os.path.exists(FRAMES_DIR):
        print(f"✗ Data directory not found: {FRAMES_DIR}")
        return

    # Create save directory if it doesn't exist
    os.makedirs(SAVE_DIR, exist_ok=True)

    # Process all crime folders
    results = process_all_crime_folders(api_key)

    # Print summary
    total_frames_processed = 0
    total_videos_processed = len(results)
    total_iterations_run = 0
    convergence_achieved = 0

    for video_id, video_results in results.items():
        if video_results and 'True_Iterative_Analysis' in video_results:
            analysis = video_results['True_Iterative_Analysis']
            total_frames_processed += analysis.get('valid_frames', 0)
            if 'convergence_summary' in analysis:
                total_iterations_run += analysis['convergence_summary'].get('total_iterations_run', 0)
                if analysis['convergence_summary'].get('converged', False):
                    convergence_achieved += 1

    print("\n" + "="*85)
    print(f"TRUE ITERATIVE PROMPTING ANALYSIS COMPLETE!")
    print(f"Videos processed: {total_videos_processed}")
    print(f"Total frames analyzed: {total_frames_processed}")
    print(f"Total iterations run: {total_iterations_run}")
    print(f"Convergence achieved: {convergence_achieved}/{total_videos_processed} videos")
    print(f"Model used: claude-opus-4-7")
    print(f"Method: Same core question refined repeatedly")
    print(f"Frame processing: {'ALL frames' if FRAME_INTERVAL == 1 else f'Every {FRAME_INTERVAL}th frame'}")
    print("=" * 85)

if __name__ == "__main__":
    run()

#Self-Consistency
This approach generates multiple independent analyses and determines the most reliable interpretation through consensus.
Self-Consistency Prompting Approach
The Self-Consistency technique follows a unique multi-analysis process:

Multiple Independent Analyses: The system generates several different analyses of the same frames
Diverse Perspectives: Each analysis uses a different prompt template to encourage varied viewpoints
Consensus Determination: The system identifies areas of agreement and disagreement across analyses
Confidence Assessment: For each key element, the level of consensus is explicitly evaluated

Implementation Highlights

Multi-Perspective Analysis:

Generates 5 independent analyses for each chunk of frames
Uses 5 distinct prompt templates to encourage diversity:

Standard analytical perspective
Forensic analyst perspective
Detective/law enforcement perspective
Security expert perspective
Witness testimony perspective


Higher temperature settings (0.5) for greater response diversity


Complete Frame Processing:

Processes all frames in chunks of 10 frames each
Each chunk undergoes the full multi-analysis process


Two-Level Consensus Building:

Chunk-Level Consensus: After generating multiple analyses for each chunk, determines consensus on:

Crime type
Perpetrator description
Victim description
Key actions
Evidence
Timeline


Cross-Chunk Consensus: After processing all chunks, synthesizes a final consensus across the entire video

In [ ]:
import os
import json
import base64
import time
from datetime import datetime
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import anthropic


import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
# ================================================================
#  CONFIGURATION  -  edit these paths to match your machine
# ================================================================
FRAMES_DIR     = r"C:\Opeyemi\PROMPTS\FRAMES"   # pre-extracted frames
RESULTS_BASE   = r"C:\Opeyemi\PROMPTS\RESULTS"  # all JSON outputs
FRAME_EXT      = ".jpg"
FRAME_INTERVAL = 1   # 1=every frame; 2=every other; etc.
MAX_WORKERS    = 8    # parallel videos processed at once (tune to API rate-limit)
BATCH_SIZE     = 20   # frames per API call (per-cell so cells run independently)

# ================================================================
#  FRAME HELPERS  -  self-contained in every cell
# ================================================================

def extract_frame_number(filename):
    """Return integer index from frame_00042.jpg style names."""
    import re as _re
    name = os.path.splitext(filename)[0]
    m = _re.search(r"frame[_\-]?(\d+)", name, _re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = _re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 0


def discover_all_videos_and_frames(frames_dir=None):
    """
    Walk FRAMES_DIR and return a manifest of all extracted videos.

    Expected layout (created by the Video-to-Frames extractor):
        FRAMES_DIR/
            Abuse/
                Abuse001_x264/
                    frame_00001.jpg
                    frame_00002.jpg ...
                Abuse002_x264/ ...
            Arrest/ ...   (13 UCF-Crime categories)

    Returns dict "<CrimeType>_<VideoStem>" -> {
        "crime_type": "Abuse",
        "video_id":   "Abuse001_x264",
        "frames_dir": r"C:\...\FRAMES\Abuse\Abuse001_x264",
        "frames":     ["frame_00001.jpg", ...]   # sorted by number
    }
    """
    if frames_dir is None:
        frames_dir = FRAMES_DIR
    print(f"\n=== DISCOVERING FRAMES ===")
    print(f"    Root : {frames_dir}")
    all_videos = {}
    if not os.path.isdir(frames_dir):
        print(f"  ERROR: FRAMES_DIR not found: {frames_dir}")
        print("  Run the Video-to-Frames extractor first, or check the path.")
        return all_videos
    crime_types = sorted([
        d for d in os.listdir(frames_dir)
        if os.path.isdir(os.path.join(frames_dir, d)) and not d.startswith("_")
    ])
    print(f"  Categories : {crime_types}")
    for crime_type in crime_types:
        cat_dir = os.path.join(frames_dir, crime_type)
        video_stems = sorted([
            d for d in os.listdir(cat_dir)
            if os.path.isdir(os.path.join(cat_dir, d))
        ])
        print(f"    {crime_type:20s}: {len(video_stems)} videos")
        for video_stem in video_stems:
            vdir = os.path.join(cat_dir, video_stem)
            frame_files = sorted(
                [ff for ff in os.listdir(vdir) if ff.lower().endswith(FRAME_EXT)],
                key=extract_frame_number
            )
            if not frame_files:
                print(f"      WARNING: no {FRAME_EXT} frames in {vdir} - skipping")
                continue
            key = f"{crime_type}_{video_stem}"
            all_videos[key] = {
                "crime_type" : crime_type,
                "video_id"   : video_stem,
                "frames_dir" : vdir,
                "frames"     : frame_files,
            }
    print(f"  Total videos ready: {len(all_videos)}")
    return all_videos


def load_frames_for_video(video_info, frame_interval=1):
    """Read every frame_interval-th .jpg, base64-encode, return ordered list of (filename, b64)."""
    vdir        = video_info["frames_dir"]
    frame_files = video_info["frames"]
    video_id    = video_info["video_id"]
    selected    = frame_files[::frame_interval]
    label = "ALL" if frame_interval == 1 else f"every {frame_interval}th"
    print(f"  Loading {len(selected)} frames ({label}) for {video_id} ...")
    frames_list = []
    for ff in selected:
        fp = os.path.join(vdir, ff)
        try:
            with open(fp, "rb") as fh:
                b64 = base64.b64encode(fh.read()).decode("utf-8")
            frames_list.append((ff, b64))
        except Exception as e:
            print(f"    ERROR loading {ff}: {e}")
    print(f"  Loaded {len(frames_list)}/{len(selected)} frames OK")
    return frames_list   # list of (filename, b64_string)

SAVE_DIR = r"C:\Opeyemi\PROMPTS\RESULTS\CLAUDE\SELF-CONSISTENCY"
os.makedirs(SAVE_DIR, exist_ok=True)


CHECKPOINT_FILE = os.path.join(SAVE_DIR, "self_consistency_checkpoint.json")


def load_checkpoint():
    """Resume from last completed video after any crash or network failure."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r") as f:
                data = json.load(f)
            n = len(data.get("completed_videos", []))
            print(f"  Checkpoint: {n} videos already done - skipping them.")
            return data
        except Exception as e:
            print(f"  Could not read checkpoint ({e}) - starting fresh.")
    return {"completed_videos": [], "results": {}}


def save_checkpoint(data):
    """Atomic write so the file is never corrupted on a crash."""
    os.makedirs(SAVE_DIR, exist_ok=True)
    tmp = CHECKPOINT_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)







def make_claude_request_robust(client, model_name, messages,
                               system=None, temperature=0.1,
                               max_retries=7, base_wait=5):
    """
    Fault-tolerant Claude API call.
    Retries on: RateLimitError, APIConnectionError (network drop/DNS),
                APITimeoutError, and 5xx server errors.
    Uses exponential back-off with jitter.
    Returns an error string on permanent failure so the caller can save it
    and move on rather than crashing the whole run.
    """
    import random, anthropic

    RETRYABLE = (
        anthropic.RateLimitError,
        anthropic.APIConnectionError,
        anthropic.APITimeoutError,
    )
    PERMANENT = (
        anthropic.AuthenticationError,
        anthropic.PermissionDeniedError,
        anthropic.NotFoundError,
    )

    for attempt in range(1, max_retries + 1):
        try:
            kwargs = dict(
                model=model_name,
                max_tokens=4096,
                temperature=temperature,
                messages=messages,
            )
            if system:
                kwargs["system"] = system
            response = client.messages.create(**kwargs)
            return response.content[0].text

        except PERMANENT as e:
            msg = f"FATAL_ERROR: {type(e).__name__}: {e}"
            print(f"[FATAL] {msg}  -- will not retry.")
            return msg

        except RETRYABLE as e:
            wait = min(base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2), 300)
            print(f"[Retry {attempt}/{max_retries}] {type(e).__name__}: {e}")
            print(f"  Waiting {wait:.1f}s before next attempt ...")
            time.sleep(wait)

        except anthropic.APIStatusError as e:
            if e.status_code >= 500:
                wait = min(base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2), 300)
                print(f"[Retry {attempt}/{max_retries}] HTTP {e.status_code}: {e}")
                print(f"  Waiting {wait:.1f}s ...")
                time.sleep(wait)
            else:
                msg = f"FATAL_ERROR: HTTP_{e.status_code}: {e}"
                print(f"[FATAL] {msg}  -- will not retry.")
                return msg

        except Exception as e:
            if attempt < max_retries:
                print(f"[Retry {attempt}/{max_retries}] Unexpected {type(e).__name__}: {e}")
                time.sleep(base_wait * attempt)
            else:
                return f"ERROR: {type(e).__name__}: {e}"

    return f"ERROR: All {max_retries} attempts exhausted"





class SelfConsistencyClaudeAnalyzer:
    """
    Self-consistency prompting: sample N independent reasoning chains at
    nonzero temperature, then aggregate by majority vote on the final
    classification. The aggregate is more robust to single-sample noise
    than any individual chain.
    """

    MODEL = "claude-opus-4-7"

    SYSTEM_PROMPT = (
        "You are an expert forensic video analyst specializing in crime detection "
        "and security surveillance. You analyze video frames methodically, noting "
        "details about people, actions, environment, and potential criminal activity. "
        "Be precise, objective, and thorough."
    )

    CRIME_LABELS = ("Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion",
                    "Fighting", "RoadAccidents", "Robbery", "Shooting",
                    "Shoplifting", "Stealing", "Vandalism", "Normal")

    def __init__(self, api_key: str, batch_size: int = BATCH_SIZE):
        self.client      = anthropic.Anthropic(api_key=api_key)
        self.batch_size  = batch_size
        self.n_samples   = 5     # independent reasoning chains per batch
        self.temperature = 0.7   # nonzero so samples diverge

    # ----------------------------------------------------------
    def _build_image_blocks(self, batch: list) -> list:
        blocks = []
        for fname, b64 in batch:
            blocks.append({
                "type": "image",
                "source": {
                    "type":       "base64",
                    "media_type": "image/jpeg",
                    "data":       b64,
                },
            })
            blocks.append({"type": "text", "text": f"[Frame: {fname}]"})
        return blocks

    # ----------------------------------------------------------
    def _one_sample(self, batch, batch_num, total_batches, sample_idx):
        image_blocks = self._build_image_blocks(batch)
        labels = ", ".join(self.CRIME_LABELS)
        framing = (
            f"Batch {batch_num}/{total_batches}, sample {sample_idx}/{self.n_samples}.\n\n"
            "Provide a step-by-step reasoning chain analyzing these video frames "
            "for criminal activity. Cover: people present, actions, interactions, "
            "objects, environment, and any indicators of crime. Then end your "
            "response with EXACTLY this final line:\n\n"
            f"FINAL_LABEL: <one of: {labels}>\n"
            "CONFIDENCE: <0-100>"
        )
        messages = [{"role": "user",
                     "content": image_blocks + [{"type": "text", "text": framing}]}]
        return make_claude_request_robust(
            self.client, self.MODEL, messages,
            system=self.SYSTEM_PROMPT, temperature=self.temperature)

    # ----------------------------------------------------------
    def _extract_label(self, text: str) -> tuple:
        """Pull (label, confidence) from the structured tail of the response."""
        import re as _re
        label_match = _re.search(r"FINAL_LABEL:\s*([A-Za-z]+)", text)
        conf_match  = _re.search(r"CONFIDENCE:\s*(\d+)",        text)
        label = label_match.group(1) if label_match else "Unknown"
        # Snap to canonical casing if it matches a known label case-insensitively
        for canon in self.CRIME_LABELS:
            if label.lower() == canon.lower():
                label = canon
                break
        conf = int(conf_match.group(1)) if conf_match else 0
        return label, conf

    # ----------------------------------------------------------
    def _vote(self, samples_with_labels):
        """Majority vote on label, confidence-weighted as tiebreaker."""
        from collections import Counter
        votes = Counter(lbl for lbl, _conf, _txt in samples_with_labels)
        # Order by (count desc, total confidence desc)
        ranked = sorted(
            votes.items(),
            key=lambda kv: (
                kv[1],
                sum(c for l, c, _ in samples_with_labels if l == kv[0]),
            ),
            reverse=True,
        )
        winner_label, winner_count = ranked[0]
        winner_confs = [c for l, c, _ in samples_with_labels if l == winner_label]
        avg_conf = sum(winner_confs) / len(winner_confs) if winner_confs else 0
        return winner_label, winner_count, avg_conf, dict(votes)

    # ----------------------------------------------------------
    def analyze_frames(self, frames_list: list, video_id: str, crime_type: str) -> dict:
        total_frames  = len(frames_list)
        batches       = [frames_list[i:i + self.batch_size]
                         for i in range(0, total_frames, self.batch_size)]
        total_batches = len(batches)
        print(f"\n  [Self-Consistency] {video_id} | {total_frames} frames | "
              f"{total_batches} batches | n_samples={self.n_samples}")

        per_batch_results = []
        for b_idx, batch in enumerate(batches, start=1):
            print(f"    Batch {b_idx}/{total_batches} - drawing {self.n_samples} samples ...")
            samples_with_labels = []
            for s_idx in range(1, self.n_samples + 1):
                txt = self._one_sample(batch, b_idx, total_batches, s_idx)
                lbl, conf = self._extract_label(txt)
                samples_with_labels.append((lbl, conf, txt))
                print(f"      sample {s_idx}: {lbl} ({conf}%)")
            winner, count, avg_conf, vote_dist = self._vote(samples_with_labels)
            print(f"      -> winner: {winner} ({count}/{self.n_samples}, avg conf {avg_conf:.0f})")
            per_batch_results.append({
                "batch_num":      b_idx,
                "samples":        [{"label": l, "confidence": c, "reasoning": t}
                                   for l, c, t in samples_with_labels],
                "winner_label":   winner,
                "winner_count":   count,
                "winner_avg_conf": avg_conf,
                "vote_distribution": vote_dist,
            })

        # Aggregate across batches: majority vote on per-batch winners
        from collections import Counter
        batch_winners = Counter(r["winner_label"] for r in per_batch_results)
        final_label, final_count = batch_winners.most_common(1)[0]
        final_confs = [r["winner_avg_conf"] for r in per_batch_results
                       if r["winner_label"] == final_label]
        final_avg_conf = sum(final_confs) / len(final_confs) if final_confs else 0

        return {
            "video_id":             video_id,
            "crime_type":           crime_type,
            "frames_analyzed":      total_frames,
            "total_batches":        total_batches,
            "batch_size":           self.batch_size,
            "prompting_technique":  "SELF-CONSISTENCY",
            "model":                self.MODEL,
            "n_samples":            self.n_samples,
            "temperature":          self.temperature,
            "timestamp":            time.strftime("%Y-%m-%d %H:%M:%S"),
            "batch_results":        per_batch_results,
            "final_label":          final_label,
            "final_batch_votes":    final_count,
            "final_avg_confidence": final_avg_conf,
            "batch_vote_distribution": dict(batch_winners),
        }


def process_all_crime_folders(api_key):
    """
    Analyse every video in FRAMES_DIR.
    Already-completed videos are skipped (checkpoint/resume) so re-running
    after any failure picks up exactly where it stopped.
    """
    analyzer   = SelfConsistencyClaudeAnalyzer(api_key)
    all_videos = discover_all_videos_and_frames()
    if not all_videos:
        print("No videos found! Verify FRAMES_DIR path."); return {}
    cp          = load_checkpoint()
    all_results = cp.get("results", {})
    done_set    = set(cp.get("completed_videos", []))
    skipped     = []
    total       = len(all_videos)
    remaining   = {k: v for k, v in all_videos.items() if k not in done_set}
    print(f"\nVideos: total={total} | done={len(done_set)} | remaining={len(remaining)}")
    # ================================================================
    #  PARALLEL VIDEO PROCESSING
    # ================================================================
    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(vkey, vinfo):
        """Process a single video in its own thread. Checkpoint writes are locked."""
        with print_lock:
            print(f"\n  [START] {vkey}")
        try:
            frames = load_frames_for_video(vinfo, FRAME_INTERVAL)
            if not frames:
                return vkey, None, "no frames"
            res = analyzer.analyze_frames(frames, vinfo["video_id"], vinfo["crime_type"])
            with checkpoint_lock:
                all_results[vkey] = res
                done_set.add(vkey)
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            with print_lock:
                print(f"  [DONE]  {vkey}  ({len(done_set)}/{total})")
            return vkey, res, None
        except Exception as e:
            with print_lock:
                print(f"  [ERROR] {vkey}: {e}")
            with checkpoint_lock:
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            return vkey, None, f"error: {e}"

    print(f"\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_process_one_video, k, v) for k, v in remaining.items()]
        for fut in as_completed(futures):
            vkey, _, err = fut.result()
            if err:
                skipped.append(f"{vkey} ({err})")

    ts = time.strftime("%Y%m%d_%H%M%S")
    summary = os.path.join(SAVE_DIR, f"self_consistency_summary_{ts}.json")
    with open(summary, "w") as f:
        json.dump(all_results, f, indent=2)
    if skipped:
        with open(os.path.join(SAVE_DIR, f"skipped_{ts}.txt"), "w") as f:
            f.write("\n".join(skipped))
    print(f"\nDone. Summary -> {summary}")
    print(f"  Processed: {len(all_results)} | Skipped: {len(skipped)}")
    return all_results


def test_claude_api(api_key):
    """Test Claude API connection"""
    print("Testing Claude API connection...")

    try:
        client = anthropic.Anthropic(api_key=api_key)

        response = client.messages.create(
            model="claude-opus-4-7",
            max_tokens=100,
            messages=[
                {
                    "role": "user",
                    "content": "Hello, can you respond with 'API connection successful'?"
                }
            ]
        )

        response_text = response.content[0].text
        print("✓ Claude API connection successful!")
        print(f"Response: {response_text}")
        return True

    except anthropic.APIError as e:
        print(f"✗ API Error: {str(e)}")
        return False
    except anthropic.AuthenticationError as e:
        print(f"✗ Authentication Error: {str(e)}")
        print("Please check your API key is valid and has sufficient credits.")
        return False
    except anthropic.RateLimitError as e:
        print(f"✗ Rate Limit Error: {str(e)}")
        return False
    except Exception as e:
        print(f"✗ Connection error: {str(e)}")
        return False

def check_authentication():
    """Placeholder function to check authentication"""
    return True

def run():
    # Load API key from file
    with open(r"C:\Opeyemi\PROMPTS\API-KEYS\claude.txt", "r") as _f:
        api_key = _f.read().strip()

    """Main execution function"""
    print("Self-Consistency Prompting Crime Video Analysis with Claude Sonnet 4 - ALL Frames")
    print("="*85)
    print("Self-Consistency = Multiple independent analyses with consistency verification")
    print("="*85)

    # Test directory access first
    print("Testing directory access...")
    for path in [FRAMES_DIR, SAVE_DIR]:
        print(f"Path: {path}")
        print(f"  Exists: {os.path.exists(path)}")
        if os.path.exists(path):
            try:
                contents = os.listdir(path)
                print(f"  Contains {len(contents)} items")
                if contents:
                    print(f"  First few items: {contents[:3]}")
            except Exception as e:
                print(f"  Error accessing contents: {str(e)}")

    # Get API key
    try:
        api_key_path = "C:\Opeyemi\PROMPTS\API-KEYS\claude.txt"
        print(f"Trying to load API key from: {api_key_path}")
        print(f"File exists: {os.path.exists(api_key_path)}")

        with open(api_key_path, "r") as f:
            api_key = f.read().strip()

        if not api_key:
            print("✗ Failed to load Claude API key: File is empty")
            return

        print("✓ Successfully loaded Claude API key")
        print(f"API key starts with: {api_key[:10]}...")

    except Exception as e:
        print(f"✗ Failed to load Claude API key: {str(e)}")
        return

    # Test Claude API connection
    if not test_claude_api(api_key):
        print("✗ Claude API test failed. Please check your API key and connection.")
        return

    # Check authentication
    if not check_authentication():
        print("✗ Authentication not completed.")
        return

    # Verify directories exist
    print("\nVerifying directories:")
    print(f"Data directory exists: {os.path.exists(FRAMES_DIR)}")
    print(f"Save directory exists: {os.path.exists(SAVE_DIR)}")

    if not os.path.exists(FRAMES_DIR):
        print(f"✗ Data directory not found: {FRAMES_DIR}")
        return

    # Create save directory if it doesn't exist
    os.makedirs(SAVE_DIR, exist_ok=True)

    # Process all crime folders
    results = process_all_crime_folders(api_key)

    # Print summary
    total_frames_processed = 0
    total_videos_processed = len(results)
    total_independent_runs = 0

    for video_id, video_results in results.items():
        if video_results and 'Self_Consistency_Analysis' in video_results:
            analysis = video_results['Self_Consistency_Analysis']
            total_frames_processed += analysis.get('valid_frames', 0)
            if 'consistency_results' in analysis and 'methodology' in analysis['consistency_results']:
                total_independent_runs += analysis['consistency_results']['methodology'].get('num_runs', 0)

    print("\n" + "="*85)
    print(f"SELF-CONSISTENCY PROMPTING ANALYSIS COMPLETE!")
    print(f"Videos processed: {total_videos_processed}")
    print(f"Total frames analyzed: {total_frames_processed}")
    print(f"Total independent runs: {total_independent_runs}")
    print(f"Model used: claude-opus-4-7")
    print(f"Analysis pattern: Multiple Independent → Consistency Check → Consensus")
    print(f"Frame processing: {'ALL frames' if FRAME_INTERVAL == 1 else f'Every {FRAME_INTERVAL}th frame'}")
    print("=" * 85)

if __name__ == "__main__":
    run()

#Meta-Prompting
- Meta-Prompting that processes all frames from crime videos. This technique is unique because it uses the AI to generate its own specialized prompts for analysis.

Meta-Prompting Approach
The Meta-Prompting technique follows this innovative process:

- Prompt Generation: Instead of using predefined prompts, the system asks the AI to create specialized prompts for analyzing video frames
- Prompt Application: These AI-generated prompts are then used to analyze the actual frames
- Meta-Synthesis: The system also generates a specialized synthesis prompt to combine all chunk analyses

Implementation Highlights

Two-Stage Meta-Prompting:

- First Stage: For each chunk of frames, generate a specialized analysis prompt
- Second Stage: For final synthesis, generate a specialized synthesis prompt
- Both stages use the AI to create task-specific prompts rather than using predefined ones


Complete Frame Processing:

- Processes all frames in chunks of 10 frames each
- Each chunk undergoes the full meta-prompting process independently


Specialized Prompt Design Process: Guides the AI to create prompts that focus on:

Step-by-step observation:
- Objective description before interpretation
- Attention to easily missed details
- Organizing observations into a coherent narrative
- Avoids including example responses in the generated prompts


Fallback Safety:

- If meta-prompting fails, falls back to a simple seed prompt
Ensures analysis can continue even if prompt generation has issues

In [ ]:
import os
import json
import base64
import time
from datetime import datetime
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import anthropic


import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
# ================================================================
#  CONFIGURATION  -  edit these paths to match your machine
# ================================================================
FRAMES_DIR     = r"C:\Opeyemi\PROMPTS\FRAMES"   # pre-extracted frames
RESULTS_BASE   = r"C:\Opeyemi\PROMPTS\RESULTS"  # all JSON outputs
FRAME_EXT      = ".jpg"
FRAME_INTERVAL = 1   # 1=every frame; 2=every other; etc.
MAX_WORKERS    = 8    # parallel videos processed at once (tune to API rate-limit)
BATCH_SIZE     = 20   # frames per API call (per-cell so cells run independently)

# ================================================================
#  FRAME HELPERS  -  self-contained in every cell
# ================================================================

def extract_frame_number(filename):
    """Return integer index from frame_00042.jpg style names."""
    import re as _re
    name = os.path.splitext(filename)[0]
    m = _re.search(r"frame[_\-]?(\d+)", name, _re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = _re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 0


def discover_all_videos_and_frames(frames_dir=None):
    """
    Walk FRAMES_DIR and return a manifest of all extracted videos.

    Expected layout (created by the Video-to-Frames extractor):
        FRAMES_DIR/
            Abuse/
                Abuse001_x264/
                    frame_00001.jpg
                    frame_00002.jpg ...
                Abuse002_x264/ ...
            Arrest/ ...   (13 UCF-Crime categories)

    Returns dict "<CrimeType>_<VideoStem>" -> {
        "crime_type": "Abuse",
        "video_id":   "Abuse001_x264",
        "frames_dir": r"C:\...\FRAMES\Abuse\Abuse001_x264",
        "frames":     ["frame_00001.jpg", ...]   # sorted by number
    }
    """
    if frames_dir is None:
        frames_dir = FRAMES_DIR
    print(f"\n=== DISCOVERING FRAMES ===")
    print(f"    Root : {frames_dir}")
    all_videos = {}
    if not os.path.isdir(frames_dir):
        print(f"  ERROR: FRAMES_DIR not found: {frames_dir}")
        print("  Run the Video-to-Frames extractor first, or check the path.")
        return all_videos
    crime_types = sorted([
        d for d in os.listdir(frames_dir)
        if os.path.isdir(os.path.join(frames_dir, d)) and not d.startswith("_")
    ])
    print(f"  Categories : {crime_types}")
    for crime_type in crime_types:
        cat_dir = os.path.join(frames_dir, crime_type)
        video_stems = sorted([
            d for d in os.listdir(cat_dir)
            if os.path.isdir(os.path.join(cat_dir, d))
        ])
        print(f"    {crime_type:20s}: {len(video_stems)} videos")
        for video_stem in video_stems:
            vdir = os.path.join(cat_dir, video_stem)
            frame_files = sorted(
                [ff for ff in os.listdir(vdir) if ff.lower().endswith(FRAME_EXT)],
                key=extract_frame_number
            )
            if not frame_files:
                print(f"      WARNING: no {FRAME_EXT} frames in {vdir} - skipping")
                continue
            key = f"{crime_type}_{video_stem}"
            all_videos[key] = {
                "crime_type" : crime_type,
                "video_id"   : video_stem,
                "frames_dir" : vdir,
                "frames"     : frame_files,
            }
    print(f"  Total videos ready: {len(all_videos)}")
    return all_videos


def load_frames_for_video(video_info, frame_interval=1):
    """Read every frame_interval-th .jpg, base64-encode, return ordered list of (filename, b64)."""
    vdir        = video_info["frames_dir"]
    frame_files = video_info["frames"]
    video_id    = video_info["video_id"]
    selected    = frame_files[::frame_interval]
    label = "ALL" if frame_interval == 1 else f"every {frame_interval}th"
    print(f"  Loading {len(selected)} frames ({label}) for {video_id} ...")
    frames_list = []
    for ff in selected:
        fp = os.path.join(vdir, ff)
        try:
            with open(fp, "rb") as fh:
                b64 = base64.b64encode(fh.read()).decode("utf-8")
            frames_list.append((ff, b64))
        except Exception as e:
            print(f"    ERROR loading {ff}: {e}")
    print(f"  Loaded {len(frames_list)}/{len(selected)} frames OK")
    return frames_list   # list of (filename, b64_string)

SAVE_DIR = r"C:\Opeyemi\PROMPTS\RESULTS\CLAUDE\META-PROMPTING"
os.makedirs(SAVE_DIR, exist_ok=True)


CHECKPOINT_FILE = os.path.join(SAVE_DIR, "meta_prompting_checkpoint.json")


def load_checkpoint():
    """Resume from last completed video after any crash or network failure."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r") as f:
                data = json.load(f)
            n = len(data.get("completed_videos", []))
            print(f"  Checkpoint: {n} videos already done - skipping them.")
            return data
        except Exception as e:
            print(f"  Could not read checkpoint ({e}) - starting fresh.")
    return {"completed_videos": [], "results": {}}


def save_checkpoint(data):
    """Atomic write so the file is never corrupted on a crash."""
    os.makedirs(SAVE_DIR, exist_ok=True)
    tmp = CHECKPOINT_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)







def make_claude_request_robust(client, model_name, messages,
                               system=None, temperature=0.1,
                               max_retries=7, base_wait=5):
    """
    Fault-tolerant Claude API call.
    Retries on: RateLimitError, APIConnectionError (network drop/DNS),
                APITimeoutError, and 5xx server errors.
    Uses exponential back-off with jitter.
    Returns an error string on permanent failure so the caller can save it
    and move on rather than crashing the whole run.
    """
    import random, anthropic

    RETRYABLE = (
        anthropic.RateLimitError,
        anthropic.APIConnectionError,
        anthropic.APITimeoutError,
    )
    PERMANENT = (
        anthropic.AuthenticationError,
        anthropic.PermissionDeniedError,
        anthropic.NotFoundError,
    )

    for attempt in range(1, max_retries + 1):
        try:
            kwargs = dict(
                model=model_name,
                max_tokens=4096,
                messages=messages,
            )
            if system:
                kwargs["system"] = system
            response = client.messages.create(**kwargs)
            return response.content[0].text

        except PERMANENT as e:
            msg = f"FATAL_ERROR: {type(e).__name__}: {e}"
            print(f"[FATAL] {msg}  -- will not retry.")
            return msg

        except RETRYABLE as e:
            wait = min(base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2), 300)
            print(f"[Retry {attempt}/{max_retries}] {type(e).__name__}: {e}")
            print(f"  Waiting {wait:.1f}s before next attempt ...")
            time.sleep(wait)

        except anthropic.APIStatusError as e:
            if e.status_code >= 500:
                wait = min(base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2), 300)
                print(f"[Retry {attempt}/{max_retries}] HTTP {e.status_code}: {e}")
                print(f"  Waiting {wait:.1f}s ...")
                time.sleep(wait)
            else:
                msg = f"FATAL_ERROR: HTTP_{e.status_code}: {e}"
                print(f"[FATAL] {msg}  -- will not retry.")
                return msg

        except Exception as e:
            if attempt < max_retries:
                print(f"[Retry {attempt}/{max_retries}] Unexpected {type(e).__name__}: {e}")
                time.sleep(base_wait * attempt)
            else:
                return f"ERROR: {type(e).__name__}: {e}"

    return f"ERROR: All {max_retries} attempts exhausted"





class MetaPromptingClaudeAnalyzer:
    """
    Meta-prompting: the model first GENERATES the analysis prompt it would
    want for this specific scene (after a quick look), then ANSWERS that
    self-generated prompt against the same frames. Two stages per batch:
      Stage 1 (planner): "What's the best prompt for analyzing these frames?"
      Stage 2 (executor): use the generated prompt to produce the final analysis.
    """

    MODEL = "claude-opus-4-7"

    SYSTEM_PROMPT = (
        "You are an expert forensic video analyst specializing in crime detection "
        "and security surveillance. You also have strong skills in prompt "
        "engineering: when asked, you can design analysis prompts that elicit "
        "thorough, evidence-grounded reasoning."
    )

    META_INSTRUCTIONS = (
        "Look briefly at these video frames. Do NOT analyze them in detail yet. "
        "Instead, design the optimal analysis prompt that a forensic video "
        "analyst should follow to determine whether a crime is occurring in "
        "this specific scene. The prompt should:\n"
        "  - be tailored to what you actually see (lighting, setting, number of people, etc.),\n"
        "  - call out the most diagnostic observations to make,\n"
        "  - require a final classification from: Abuse, Arrest, Arson, Assault, "
        "Burglary, Explosion, Fighting, RoadAccidents, Robbery, Shooting, "
        "Shoplifting, Stealing, Vandalism, Normal,\n"
        "  - require a confidence score and key evidence.\n\n"
        "Output ONLY the prompt itself, no preamble, no explanation. Start "
        "directly with the prompt text."
    )

    def __init__(self, api_key: str, batch_size: int = BATCH_SIZE):
        self.client     = anthropic.Anthropic(api_key=api_key)
        self.batch_size = batch_size

    # ----------------------------------------------------------
    def _build_image_blocks(self, batch: list) -> list:
        blocks = []
        for fname, b64 in batch:
            blocks.append({
                "type": "image",
                "source": {
                    "type":       "base64",
                    "media_type": "image/jpeg",
                    "data":       b64,
                },
            })
            blocks.append({"type": "text", "text": f"[Frame: {fname}]"})
        return blocks

    # ----------------------------------------------------------
    def _generate_prompt(self, batch, batch_num, total_batches):
        image_blocks = self._build_image_blocks(batch)
        framing = (
            f"Batch {batch_num}/{total_batches}. Stage 1 of 2: META-PROMPT GENERATION.\n\n"
            f"{self.META_INSTRUCTIONS}"
        )
        messages = [{"role": "user",
                     "content": image_blocks + [{"type": "text", "text": framing}]}]
        return make_claude_request_robust(
            self.client, self.MODEL, messages, system=self.SYSTEM_PROMPT)

    def _execute_prompt(self, batch, batch_num, total_batches, generated_prompt):
        image_blocks = self._build_image_blocks(batch)
        framing = (
            f"Batch {batch_num}/{total_batches}. Stage 2 of 2: EXECUTE THE GENERATED PROMPT.\n\n"
            f"Below is the analysis prompt designed for this specific scene. "
            f"Follow it precisely:\n\n"
            f"--- BEGIN GENERATED PROMPT ---\n{generated_prompt}\n--- END GENERATED PROMPT ---"
        )
        messages = [{"role": "user",
                     "content": image_blocks + [{"type": "text", "text": framing}]}]
        return make_claude_request_robust(
            self.client, self.MODEL, messages, system=self.SYSTEM_PROMPT)

    # ----------------------------------------------------------
    def _synthesize(self, per_batch, total_frames, total_batches):
        joined = "\n\n".join(
            f"--- Batch {b['batch_num']} executed analysis ---\n{b['executed_analysis']}"
            for b in per_batch
        )
        synth_prompt = (
            f"You used meta-prompting to analyze {total_batches} batches covering "
            f"{total_frames} frames of one security video. Below is each batch's "
            f"executed analysis:\n\n{joined}\n\n"
            "Produce a FINAL combined report:\n"
            "1. SCENE DESCRIPTION.\n"
            "2. CRIME CLASSIFICATION (Abuse, Arrest, Arson, Assault, Burglary, "
            "Explosion, Fighting, RoadAccidents, Robbery, Shooting, Shoplifting, "
            "Stealing, Vandalism, Normal).\n"
            "3. CONFIDENCE (0-100%).\n"
            "4. KEY EVIDENCE.\n"
            "5. RECOMMENDED ACTION."
        )
        return make_claude_request_robust(
            self.client, self.MODEL,
            [{"role": "user", "content": synth_prompt}],
            system=self.SYSTEM_PROMPT)

    # ----------------------------------------------------------
    def analyze_frames(self, frames_list: list, video_id: str, crime_type: str) -> dict:
        total_frames  = len(frames_list)
        batches       = [frames_list[i:i + self.batch_size]
                         for i in range(0, total_frames, self.batch_size)]
        total_batches = len(batches)
        print(f"\n  [Meta-Prompting] {video_id} | {total_frames} frames | "
              f"{total_batches} batches of up to {self.batch_size}")

        per_batch = []
        for b_idx, batch in enumerate(batches, start=1):
            print(f"    Batch {b_idx}/{total_batches} - generating prompt ...")
            gen_prompt = self._generate_prompt(batch, b_idx, total_batches)
            print(f"      generated prompt: {len(gen_prompt)} chars")
            print(f"    Batch {b_idx}/{total_batches} - executing prompt ...")
            executed = self._execute_prompt(batch, b_idx, total_batches, gen_prompt)
            print(f"      executed analysis: {len(executed)} chars")
            per_batch.append({
                "batch_num":         b_idx,
                "generated_prompt":  gen_prompt,
                "executed_analysis": executed,
            })

        print(f"    Synthesizing {total_batches} batches ...")
        final_report = self._synthesize(per_batch, total_frames, total_batches)
        print(f"      final report: {len(final_report)} chars")

        return {
            "video_id":             video_id,
            "crime_type":           crime_type,
            "frames_analyzed":      total_frames,
            "total_batches":        total_batches,
            "batch_size":           self.batch_size,
            "prompting_technique":  "META-PROMPTING",
            "model":                self.MODEL,
            "timestamp":            time.strftime("%Y-%m-%d %H:%M:%S"),
            "batch_traces":         per_batch,
            "final_analysis":       final_report,
        }


def process_all_crime_folders(api_key):
    """
    Analyse every video in FRAMES_DIR.
    Already-completed videos are skipped (checkpoint/resume) so re-running
    after any failure picks up exactly where it stopped.
    """
    analyzer   = MetaPromptingClaudeAnalyzer(api_key)
    all_videos = discover_all_videos_and_frames()
    if not all_videos:
        print("No videos found! Verify FRAMES_DIR path."); return {}
    cp          = load_checkpoint()
    all_results = cp.get("results", {})
    done_set    = set(cp.get("completed_videos", []))
    skipped     = []
    total       = len(all_videos)
    remaining   = {k: v for k, v in all_videos.items() if k not in done_set}
    print(f"\nVideos: total={total} | done={len(done_set)} | remaining={len(remaining)}")
    # ================================================================
    #  PARALLEL VIDEO PROCESSING
    # ================================================================
    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(vkey, vinfo):
        """Process a single video in its own thread. Checkpoint writes are locked."""
        with print_lock:
            print(f"\n  [START] {vkey}")
        try:
            frames = load_frames_for_video(vinfo, FRAME_INTERVAL)
            if not frames:
                return vkey, None, "no frames"
            res = analyzer.analyze_frames(frames, vinfo["video_id"], vinfo["crime_type"])
            with checkpoint_lock:
                all_results[vkey] = res
                done_set.add(vkey)
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            with print_lock:
                print(f"  [DONE]  {vkey}  ({len(done_set)}/{total})")
            return vkey, res, None
        except Exception as e:
            with print_lock:
                print(f"  [ERROR] {vkey}: {e}")
            with checkpoint_lock:
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            return vkey, None, f"error: {e}"

    print(f"\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_process_one_video, k, v) for k, v in remaining.items()]
        for fut in as_completed(futures):
            vkey, _, err = fut.result()
            if err:
                skipped.append(f"{vkey} ({err})")

    ts = time.strftime("%Y%m%d_%H%M%S")
    summary = os.path.join(SAVE_DIR, f"meta_prompting_summary_{ts}.json")
    with open(summary, "w") as f:
        json.dump(all_results, f, indent=2)
    if skipped:
        with open(os.path.join(SAVE_DIR, f"skipped_{ts}.txt"), "w") as f:
            f.write("\n".join(skipped))
    print(f"\nDone. Summary -> {summary}")
    print(f"  Processed: {len(all_results)} | Skipped: {len(skipped)}")
    return all_results


def test_claude_api(api_key):
    """Test Claude API connection"""
    print("Testing Claude API connection...")

    try:
        client = anthropic.Anthropic(api_key=api_key)

        response = client.messages.create(
            model="claude-opus-4-7",
            max_tokens=100,
            messages=[
                {
                    "role": "user",
                    "content": "Hello, can you respond with 'API connection successful'?"
                }
            ]
        )

        response_text = response.content[0].text
        print("✓ Claude API connection successful!")
        print(f"Response: {response_text}")
        return True

    except anthropic.APIError as e:
        print(f"✗ API Error: {str(e)}")
        return False
    except anthropic.AuthenticationError as e:
        print(f"✗ Authentication Error: {str(e)}")
        print("Please check your API key is valid and has sufficient credits.")
        return False
    except anthropic.RateLimitError as e:
        print(f"✗ Rate Limit Error: {str(e)}")
        return False
    except Exception as e:
        print(f"✗ Connection error: {str(e)}")
        return False

def check_authentication():
    """Placeholder function to check authentication"""
    return True

def run():
    # Load API key from file
    with open(r"C:\Opeyemi\PROMPTS\API-KEYS\claude.txt", "r") as _f:
        api_key = _f.read().strip()

    """Main execution function"""
    print("Meta-Prompting Crime Video Analysis with Claude Sonnet 4 - ALL Frames")
    print("="*80)
    print("Meta-Prompting = Self-generated optimized prompts for enhanced analysis")
    print("="*80)

    # Test directory access first
    print("Testing directory access...")
    for path in [FRAMES_DIR, SAVE_DIR]:
        print(f"Path: {path}")
        print(f"  Exists: {os.path.exists(path)}")
        if os.path.exists(path):
            try:
                contents = os.listdir(path)
                print(f"  Contains {len(contents)} items")
                if contents:
                    print(f"  First few items: {contents[:3]}")
            except Exception as e:
                print(f"  Error accessing contents: {str(e)}")

    # Get API key
    try:
        api_key_path = "C:\Opeyemi\PROMPTS\API-KEYS\claude.txt"
        print(f"Trying to load API key from: {api_key_path}")
        print(f"File exists: {os.path.exists(api_key_path)}")

        with open(api_key_path, "r") as f:
            api_key = f.read().strip()

        if not api_key:
            print("✗ Failed to load Claude API key: File is empty")
            return

        print("✓ Successfully loaded Claude API key")
        print(f"API key starts with: {api_key[:10]}...")

    except Exception as e:
        print(f"✗ Failed to load Claude API key: {str(e)}")
        return

    # Test Claude API connection
    if not test_claude_api(api_key):
        print("✗ Claude API test failed. Please check your API key and connection.")
        return

    # Check authentication
    if not check_authentication():
        print("✗ Authentication not completed.")
        return

    # Verify directories exist
    print("\nVerifying directories:")
    print(f"Data directory exists: {os.path.exists(FRAMES_DIR)}")
    print(f"Save directory exists: {os.path.exists(SAVE_DIR)}")

    if not os.path.exists(FRAMES_DIR):
        print(f"✗ Data directory not found: {FRAMES_DIR}")
        return

    # Create save directory if it doesn't exist
    os.makedirs(SAVE_DIR, exist_ok=True)

    # Process all crime folders
    results = process_all_crime_folders(api_key)

    # Print summary
    total_frames_processed = 0
    total_videos_processed = len(results)
    total_phases_completed = 0

    for video_id, video_results in results.items():
        if video_results and 'Meta_Prompting_Analysis' in video_results:
            analysis = video_results['Meta_Prompting_Analysis']
            total_frames_processed += analysis.get('valid_frames', 0)
            if 'meta_prompting_results' in analysis and 'methodology' in analysis['meta_prompting_results']:
                total_phases_completed += len(analysis['meta_prompting_results']['methodology'].get('phases', []))

    print("\n" + "="*80)
    print(f"META-PROMPTING ANALYSIS COMPLETE!")
    print(f"Videos processed: {total_videos_processed}")
    print(f"Total frames analyzed: {total_frames_processed}")
    print(f"Total meta-phases completed: {total_phases_completed}")
    print(f"Model used: claude-opus-4-7")
    print(f"Analysis pattern: Generate → Extract → Apply → Evaluate")
    print(f"Frame processing: {'ALL frames' if FRAME_INTERVAL == 1 else f'Every {FRAME_INTERVAL}th frame'}")
    print("=" * 80)

if __name__ == "__main__":
    run()

#Chain-Of-Thought Prompting
Chain of Thought (CoT) prompting approach that processes all frames from crime videos. This technique explicitly encourages the model to show its

step-by-step reasoning process.
- Chain of Thought Prompting Approach: The Chain of Thought technique follows this explicit reasoning process:

Step-by-Step Reasoning: The approach explicitly asks the model to "think step by step" through its analysis
- Transparent Reasoning: Each reasoning step is clearly articulated in the response
- Structured Progression: The analysis follows a logical progression from observation to conclusion
- Reasoning Synthesis: The final synthesis also uses step-by-step reasoning to connect all segments

Implementation Highlights

Structured Reasoning Steps:

The prompt breaks down the analysis into 6 clear steps:

- Objective observation without interpretation
- Identification of key actors
- Chronological sequence of events
- Important objects and their usage
- Context and setting analysis
- Integration of observations into a coherent description


Each step builds on the previous one in a logical progression


Complete Frame Processing:

- Processes all frames in chunks of 10 frames each
- Each chunk undergoes the full chain of thought process independently


Reasoning-Based Synthesis: The synthesis prompt also follows a chain of thought structure:

- Extraction of key information from each segment
- Timeline construction across all segments
- Tracking people across multiple segments
- Tracking objects across segments
- Contextual integration of segments
- Construction of a comprehensive description


This ensures the synthesis uses the same reasoning approach as individual chunks


Explicit Prompting for Reasoning:

- Both the analysis and synthesis prompts specifically ask to "think step by step"
- System messages reinforce the importance of step-by-step reasoning
The model is explicitly asked to show its thinking process at each step

In [ ]:
import os
import json
import base64
import time
from datetime import datetime
from collections import defaultdict
import anthropic


import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
# ================================================================
#  CONFIGURATION  -  edit these paths to match your machine
# ================================================================
FRAMES_DIR     = r"C:\Opeyemi\PROMPTS\FRAMES"   # pre-extracted frames
RESULTS_BASE   = r"C:\Opeyemi\PROMPTS\RESULTS"  # all JSON outputs
FRAME_EXT      = ".jpg"
FRAME_INTERVAL = 1   # 1=every frame; 2=every other; etc.
MAX_WORKERS    = 8    # parallel videos processed at once (tune to API rate-limit)
BATCH_SIZE     = 20   # frames per API call (per-cell so cells run independently)

# ================================================================
#  FRAME HELPERS  -  self-contained in every cell
# ================================================================

def extract_frame_number(filename):
    """Return integer index from frame_00042.jpg style names."""
    import re as _re
    name = os.path.splitext(filename)[0]
    m = _re.search(r"frame[_\-]?(\d+)", name, _re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = _re.findall(r"\d+", name)
    return int(nums[-1]) if nums else 0


def discover_all_videos_and_frames(frames_dir=None):
    """
    Walk FRAMES_DIR and return a manifest of all extracted videos.

    Expected layout (created by the Video-to-Frames extractor):
        FRAMES_DIR/
            Abuse/
                Abuse001_x264/
                    frame_00001.jpg
                    frame_00002.jpg ...
                Abuse002_x264/ ...
            Arrest/ ...   (13 UCF-Crime categories)

    Returns dict "<CrimeType>_<VideoStem>" -> {
        "crime_type": "Abuse",
        "video_id":   "Abuse001_x264",
        "frames_dir": r"C:\...\FRAMES\Abuse\Abuse001_x264",
        "frames":     ["frame_00001.jpg", ...]   # sorted by number
    }
    """
    if frames_dir is None:
        frames_dir = FRAMES_DIR
    print(f"\n=== DISCOVERING FRAMES ===")
    print(f"    Root : {frames_dir}")
    all_videos = {}
    if not os.path.isdir(frames_dir):
        print(f"  ERROR: FRAMES_DIR not found: {frames_dir}")
        print("  Run the Video-to-Frames extractor first, or check the path.")
        return all_videos
    crime_types = sorted([
        d for d in os.listdir(frames_dir)
        if os.path.isdir(os.path.join(frames_dir, d)) and not d.startswith("_")
    ])
    print(f"  Categories : {crime_types}")
    for crime_type in crime_types:
        cat_dir = os.path.join(frames_dir, crime_type)
        video_stems = sorted([
            d for d in os.listdir(cat_dir)
            if os.path.isdir(os.path.join(cat_dir, d))
        ])
        print(f"    {crime_type:20s}: {len(video_stems)} videos")
        for video_stem in video_stems:
            vdir = os.path.join(cat_dir, video_stem)
            frame_files = sorted(
                [ff for ff in os.listdir(vdir) if ff.lower().endswith(FRAME_EXT)],
                key=extract_frame_number
            )
            if not frame_files:
                print(f"      WARNING: no {FRAME_EXT} frames in {vdir} - skipping")
                continue
            key = f"{crime_type}_{video_stem}"
            all_videos[key] = {
                "crime_type" : crime_type,
                "video_id"   : video_stem,
                "frames_dir" : vdir,
                "frames"     : frame_files,
            }
    print(f"  Total videos ready: {len(all_videos)}")
    return all_videos


def load_frames_for_video(video_info, frame_interval=1):
    """Read every frame_interval-th .jpg, base64-encode, return ordered list of (filename, b64)."""
    vdir        = video_info["frames_dir"]
    frame_files = video_info["frames"]
    video_id    = video_info["video_id"]
    selected    = frame_files[::frame_interval]
    label = "ALL" if frame_interval == 1 else f"every {frame_interval}th"
    print(f"  Loading {len(selected)} frames ({label}) for {video_id} ...")
    frames_list = []
    for ff in selected:
        fp = os.path.join(vdir, ff)
        try:
            with open(fp, "rb") as fh:
                b64 = base64.b64encode(fh.read()).decode("utf-8")
            frames_list.append((ff, b64))
        except Exception as e:
            print(f"    ERROR loading {ff}: {e}")
    print(f"  Loaded {len(frames_list)}/{len(selected)} frames OK")
    return frames_list   # list of (filename, b64_string)

SAVE_DIR = r"C:\Opeyemi\PROMPTS\RESULTS\CLAUDE\CHAIN-OF-THOUGHT"
os.makedirs(SAVE_DIR, exist_ok=True)


CHECKPOINT_FILE = os.path.join(SAVE_DIR, "cot_checkpoint.json")


def load_checkpoint():
    """Resume from last completed video after any crash or network failure."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r") as f:
                data = json.load(f)
            n = len(data.get("completed_videos", []))
            print(f"  Checkpoint: {n} videos already done - skipping them.")
            return data
        except Exception as e:
            print(f"  Could not read checkpoint ({e}) - starting fresh.")
    return {"completed_videos": [], "results": {}}


def save_checkpoint(data):
    """Atomic write so the file is never corrupted on a crash."""
    os.makedirs(SAVE_DIR, exist_ok=True)
    tmp = CHECKPOINT_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)







def make_claude_request_robust(client, model_name, messages,
                               system=None, temperature=0.1,
                               max_retries=7, base_wait=5):
    """
    Fault-tolerant Claude API call.
    Retries on: RateLimitError, APIConnectionError (network drop/DNS),
                APITimeoutError, and 5xx server errors.
    Uses exponential back-off with jitter.
    Returns an error string on permanent failure so the caller can save it
    and move on rather than crashing the whole run.
    """
    import random, anthropic

    RETRYABLE = (
        anthropic.RateLimitError,
        anthropic.APIConnectionError,
        anthropic.APITimeoutError,
    )
    PERMANENT = (
        anthropic.AuthenticationError,
        anthropic.PermissionDeniedError,
        anthropic.NotFoundError,
    )

    for attempt in range(1, max_retries + 1):
        try:
            kwargs = dict(
                model=model_name,
                max_tokens=4096,
                messages=messages,
            )
            if system:
                kwargs["system"] = system
            response = client.messages.create(**kwargs)
            return response.content[0].text

        except PERMANENT as e:
            msg = f"FATAL_ERROR: {type(e).__name__}: {e}"
            print(f"[FATAL] {msg}  -- will not retry.")
            return msg

        except RETRYABLE as e:
            wait = min(base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2), 300)
            print(f"[Retry {attempt}/{max_retries}] {type(e).__name__}: {e}")
            print(f"  Waiting {wait:.1f}s before next attempt ...")
            time.sleep(wait)

        except anthropic.APIStatusError as e:
            if e.status_code >= 500:
                wait = min(base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2), 300)
                print(f"[Retry {attempt}/{max_retries}] HTTP {e.status_code}: {e}")
                print(f"  Waiting {wait:.1f}s ...")
                time.sleep(wait)
            else:
                msg = f"FATAL_ERROR: HTTP_{e.status_code}: {e}"
                print(f"[FATAL] {msg}  -- will not retry.")
                return msg

        except Exception as e:
            if attempt < max_retries:
                print(f"[Retry {attempt}/{max_retries}] Unexpected {type(e).__name__}: {e}")
                time.sleep(base_wait * attempt)
            else:
                return f"ERROR: {type(e).__name__}: {e}"

    return f"ERROR: All {max_retries} attempts exhausted"





class ChainOfThoughtClaudeAnalyzer:
    """
    Chain-of-Thought prompting: explicit step-by-step reasoning template
    applied to every batch. The model is instructed to walk through fixed
    reasoning stages before producing the classification, with a
    "Let's think step by step" framing.
    """

    MODEL = "claude-opus-4-7"

    SYSTEM_PROMPT = (
        "You are an expert forensic video analyst specializing in crime detection "
        "and security surveillance. You analyze video frames methodically, walking "
        "through your reasoning step by step. You always ground each conclusion in "
        "specific visual evidence from the frames."
    )

    COT_TEMPLATE = (
        "Analyze these video frames for criminal activity. Let's think step by step.\n\n"
        "Walk through these reasoning stages explicitly. Label each stage in your "
        "response:\n\n"
        "STEP 1 - OBSERVATIONS: List concrete visual facts. People (count, clothing, "
        "approximate age/build), objects (vehicles, weapons, tools, packages), "
        "environment (indoor/outdoor, lighting, location type). Just facts, no inference.\n\n"
        "STEP 2 - ACTIONS: For each person, what specific actions are they performing? "
        "How do those actions change across frames?\n\n"
        "STEP 3 - INTERACTIONS: How are people interacting with each other and with "
        "objects? Who initiates contact? Are interactions cooperative, hostile, or neutral?\n\n"
        "STEP 4 - INDICATORS: What specific elements suggest criminal vs. non-criminal "
        "activity? List supporting evidence for each side.\n\n"
        "STEP 5 - HYPOTHESIS: Given STEPS 1-4, what is the most likely explanation? "
        "What alternative explanations are also plausible, and what would distinguish them?\n\n"
        "STEP 6 - CLASSIFICATION: Choose ONE of: Abuse, Arrest, Arson, Assault, "
        "Burglary, Explosion, Fighting, RoadAccidents, Robbery, Shooting, "
        "Shoplifting, Stealing, Vandalism, Normal. Give a confidence (0-100%) and "
        "the 2-3 strongest pieces of evidence."
    )

    SYNTHESIS_TEMPLATE = (
        "You used chain-of-thought reasoning across {total_batches} batches "
        "covering {total_frames} frames of one security video. Below is each "
        "batch's complete reasoning chain:\n\n{joined}\n\n"
        "Now reason step by step ONE MORE TIME, this time across batches:\n\n"
        "STEP A - CROSS-BATCH OBSERVATIONS: What is consistent across batches? "
        "What changes?\n\n"
        "STEP B - TEMPORAL TIMELINE: Build a chronological timeline of events.\n\n"
        "STEP C - FINAL CLASSIFICATION: Choose ONE of Abuse, Arrest, Arson, Assault, "
        "Burglary, Explosion, Fighting, RoadAccidents, Robbery, Shooting, "
        "Shoplifting, Stealing, Vandalism, Normal.\n"
        "STEP D - CONFIDENCE: 0-100%.\n"
        "STEP E - KEY EVIDENCE: 3-5 specific items.\n"
        "STEP F - RECOMMENDED ACTION."
    )

    def __init__(self, api_key: str, batch_size: int = BATCH_SIZE):
        self.client     = anthropic.Anthropic(api_key=api_key)
        self.batch_size = batch_size

    # ----------------------------------------------------------
    def _build_image_blocks(self, batch: list) -> list:
        blocks = []
        for fname, b64 in batch:
            blocks.append({
                "type": "image",
                "source": {
                    "type":       "base64",
                    "media_type": "image/jpeg",
                    "data":       b64,
                },
            })
            blocks.append({"type": "text", "text": f"[Frame: {fname}]"})
        return blocks

    # ----------------------------------------------------------
    def _analyze_batch(self, batch, batch_num, total_batches):
        image_blocks = self._build_image_blocks(batch)
        framing = (
            f"Batch {batch_num}/{total_batches}.\n\n{self.COT_TEMPLATE}"
        )
        messages = [{"role": "user",
                     "content": image_blocks + [{"type": "text", "text": framing}]}]
        return make_claude_request_robust(
            self.client, self.MODEL, messages, system=self.SYSTEM_PROMPT)

    def _synthesize(self, per_batch, total_frames, total_batches):
        joined = "\n\n".join(
            f"--- Batch {b['batch_num']} reasoning chain ---\n{b['reasoning_chain']}"
            for b in per_batch
        )
        synth_prompt = self.SYNTHESIS_TEMPLATE.format(
            total_batches=total_batches, total_frames=total_frames, joined=joined,
        )
        return make_claude_request_robust(
            self.client, self.MODEL,
            [{"role": "user", "content": synth_prompt}],
            system=self.SYSTEM_PROMPT)

    # ----------------------------------------------------------
    def analyze_frames(self, frames_list: list, video_id: str, crime_type: str) -> dict:
        total_frames  = len(frames_list)
        batches       = [frames_list[i:i + self.batch_size]
                         for i in range(0, total_frames, self.batch_size)]
        total_batches = len(batches)
        print(f"\n  [Chain-of-Thought] {video_id} | {total_frames} frames | "
              f"{total_batches} batches of up to {self.batch_size}")

        per_batch = []
        for b_idx, batch in enumerate(batches, start=1):
            print(f"    Batch {b_idx}/{total_batches} ...")
            chain = self._analyze_batch(batch, b_idx, total_batches)
            print(f"      reasoning chain: {len(chain)} chars")
            per_batch.append({"batch_num": b_idx, "reasoning_chain": chain})

        print(f"    Synthesizing {total_batches} batches ...")
        final_report = self._synthesize(per_batch, total_frames, total_batches)
        print(f"      final report: {len(final_report)} chars")

        return {
            "video_id":             video_id,
            "crime_type":           crime_type,
            "frames_analyzed":      total_frames,
            "total_batches":        total_batches,
            "batch_size":           self.batch_size,
            "prompting_technique":  "CHAIN-OF-THOUGHT",
            "model":                self.MODEL,
            "timestamp":            time.strftime("%Y-%m-%d %H:%M:%S"),
            "batch_chains":         per_batch,
            "final_analysis":       final_report,
        }


def process_all_crime_folders(api_key):
    """
    Analyse every video in FRAMES_DIR.
    Already-completed videos are skipped (checkpoint/resume) so re-running
    after any failure picks up exactly where it stopped.
    """
    analyzer   = ChainOfThoughtClaudeAnalyzer(api_key)
    all_videos = discover_all_videos_and_frames()
    if not all_videos:
        print("No videos found! Verify FRAMES_DIR path."); return {}
    cp          = load_checkpoint()
    all_results = cp.get("results", {})
    done_set    = set(cp.get("completed_videos", []))
    skipped     = []
    total       = len(all_videos)
    remaining   = {k: v for k, v in all_videos.items() if k not in done_set}
    print(f"\nVideos: total={total} | done={len(done_set)} | remaining={len(remaining)}")
    # ================================================================
    #  PARALLEL VIDEO PROCESSING
    # ================================================================
    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(vkey, vinfo):
        """Process a single video in its own thread. Checkpoint writes are locked."""
        with print_lock:
            print(f"\n  [START] {vkey}")
        try:
            frames = load_frames_for_video(vinfo, FRAME_INTERVAL)
            if not frames:
                return vkey, None, "no frames"
            res = analyzer.analyze_frames(frames, vinfo["video_id"], vinfo["crime_type"])
            with checkpoint_lock:
                all_results[vkey] = res
                done_set.add(vkey)
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            with print_lock:
                print(f"  [DONE]  {vkey}  ({len(done_set)}/{total})")
            return vkey, res, None
        except Exception as e:
            with print_lock:
                print(f"  [ERROR] {vkey}: {e}")
            with checkpoint_lock:
                save_checkpoint({"completed_videos": list(done_set), "results": all_results})
            return vkey, None, f"error: {e}"

    print(f"\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_process_one_video, k, v) for k, v in remaining.items()]
        for fut in as_completed(futures):
            vkey, _, err = fut.result()
            if err:
                skipped.append(f"{vkey} ({err})")

    ts = time.strftime("%Y%m%d_%H%M%S")
    summary = os.path.join(SAVE_DIR, f"cot_summary_{ts}.json")
    with open(summary, "w") as f:
        json.dump(all_results, f, indent=2)
    if skipped:
        with open(os.path.join(SAVE_DIR, f"skipped_{ts}.txt"), "w") as f:
            f.write("\n".join(skipped))
    print(f"\nDone. Summary -> {summary}")
    print(f"  Processed: {len(all_results)} | Skipped: {len(skipped)}")
    return all_results


def test_claude_api(api_key):
    """Test Claude API connection"""
    print("Testing Claude API connection...")

    try:
        client = anthropic.Anthropic(api_key=api_key)

        response = client.messages.create(
            model="claude-opus-4-7",
            max_tokens=100,
            messages=[
                {
                    "role": "user",
                    "content": "Hello, can you respond with 'API connection successful'?"
                }
            ]
        )

        response_text = response.content[0].text
        print("✓ Claude API connection successful!")
        print(f"Response: {response_text}")
        return True

    except anthropic.APIError as e:
        print(f"✗ API Error: {str(e)}")
        return False
    except anthropic.AuthenticationError as e:
        print(f"✗ Authentication Error: {str(e)}")
        print("Please check your API key is valid and has sufficient credits.")
        return False
    except anthropic.RateLimitError as e:
        print(f"✗ Rate Limit Error: {str(e)}")
        return False
    except Exception as e:
        print(f"✗ Connection error: {str(e)}")
        return False

def run():
    # Load API key from file
    with open(r"C:\Opeyemi\PROMPTS\API-KEYS\claude.txt", "r") as _f:
        api_key = _f.read().strip()

    """Main execution function"""
    print("Chain of Thought Prompting Crime Video Analysis with Claude Sonnet 4 - ALL Frames")
    print("="*80)
    print("Chain of Thought = Step-by-step reasoning with explicit thinking process")
    print("="*80)

    # Test directory access first
    print("Testing directory access...")
    for path in [FRAMES_DIR, SAVE_DIR]:
        print(f"Path: {path}")
        print(f"  Exists: {os.path.exists(path)}")
        if os.path.exists(path):
            try:
                contents = os.listdir(path)
                print(f"  Contains {len(contents)} items")
                if contents:
                    print(f"  First few items: {contents[:3]}")
            except Exception as e:
                print(f"  Error accessing contents: {str(e)}")

    # Get API key
    try:
        api_key_path = "C:\Opeyemi\PROMPTS\API-KEYS\claude.txt"
        print(f"Trying to load API key from: {api_key_path}")
        print(f"File exists: {os.path.exists(api_key_path)}")

        with open(api_key_path, "r") as f:
            api_key = f.read().strip()

        if not api_key:
            print("✗ Failed to load Claude API key: File is empty")
            return

        print("✓ Successfully loaded Claude API key")
        print(f"API key starts with: {api_key[:10]}...")

    except Exception as e:
        print(f"✗ Failed to load Claude API key: {str(e)}")
        return

    # Test Claude API connection
    if not test_claude_api(api_key):
        print("✗ Claude API test failed. Please check your API key and connection.")
        return

    # Verify directories exist
    print("\nVerifying directories:")
    print(f"Data directory exists: {os.path.exists(FRAMES_DIR)}")
    print(f"Save directory exists: {os.path.exists(SAVE_DIR)}")

    if not os.path.exists(FRAMES_DIR):
        print(f"✗ Data directory not found: {FRAMES_DIR}")
        return

    # Create save directory if it doesn't exist
    os.makedirs(SAVE_DIR, exist_ok=True)

    # Process all crime folders
    results = process_all_crime_folders(api_key)

    # Print summary
    total_frames_processed = 0
    total_videos_processed = len(results)
    total_chunks_processed = 0

    for video_id, video_results in results.items():
        if video_results and 'cot_results' in video_results:
            total_frames_processed += video_results.get('frames_used', 0)
            total_chunks_processed += video_results.get('chunks_processed', 0)

    print("\n" + "="*80)
    print(f"CHAIN OF THOUGHT PROMPTING COMPLETE!")
    print(f"Videos processed: {total_videos_processed}")
    print(f"Total frames analyzed: {total_frames_processed}")
    print(f"Total chunks processed: {total_chunks_processed}")
    print(f"Model used: claude-opus-4-7")
    print(f"Analysis pattern: Step 1 → Step 2 → ... → Step 6 → Synthesis")
    print(f"Frame processing: {'ALL frames' if FRAME_INTERVAL == 1 else f'Every {FRAME_INTERVAL}th frame'}")
    print("=" * 80)

if __name__ == "__main__":
    run()